# 次に来るトークンの確率分布を見る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_transformers/next_token_probs.ipynb)

NLP クックブック 09-06 ／ Transformers の基本操作

言語モデルが「次に何が来る」と予測しているのか、上位候補と確率を直接見たい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
model = AutoModelForCausalLM.from_pretrained(name).eval()

def next_tokens(prefix, k=6):
    ids = tokenizer(tokenizer.bos_token + prefix, add_special_tokens=False, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logits = model(ids).logits[0, -1]            # 最後の位置の出力 = 次のトークンの予測
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    entropy = -(probs * torch.log2(probs + 1e-12)).sum().item()
    print(f"{prefix} … (エントロピー {entropy:.2f} bit)")
    print("   ", "  ".join(f"{tokenizer.decode(i)!r}:{p:.2f}" for p, i in zip(top.values.tolist(), top.indices.tolist())))

for prefix in ["日本の首都は", "むかしむかし、あるところに", "今日の天気は", "1 + 1 ="]:
    next_tokens(prefix)

日本の首都は … (エントロピー 2.17 bit)
    '東京':0.80  '大阪':0.03  '？':0.02  '、':0.01  'どこか':0.01  '「':0.01
むかしむかし、あるところに … (エントロピー 6.62 bit)
    '、':0.29  '一人の':0.22  '小さな':0.05  'おじいさん':0.02  'ひとつの':0.01  '１':0.01
今日の天気は … (エントロピー 6.16 bit)
    '晴れ':0.33  '？':0.13  '雨':0.09  'どちら':0.04  'いかがでしょうか':0.03  '?':0.02
1 + 1 = … (エントロピー 0.14 bit)
    ' ':0.99  ' "':0.00  '2':0.00  ' (':0.00  '\n':0.00  ' __':0.00


In [2]:
# 特定の候補同士の確率を比べる (選択式問題を解かせる、など)
def compare(prefix, candidates):
    ids = tokenizer(tokenizer.bos_token + prefix, add_special_tokens=False, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logp = torch.log_softmax(model(ids).logits[0, -1], dim=-1)
    return {c: round(logp[tokenizer(c, add_special_tokens=False)["input_ids"][0]].item(), 2) for c in candidates}

print(compare("富士山があるのは", ["日本", "中国", "アメリカ"]))

{'日本': -1.89, '中国': -7.31, 'アメリカ': -7.19}


- 言語モデルの出力は、語彙全体にわたる確率分布。生成とは、この分布から 1 つ選んで追加することの繰り返し。
- エントロピーが低い = 次の語がほぼ決まっている。高い = 候補が多くて迷っている。
- `compare` は各候補の最初の 1 トークンだけを比べている点に注意。複数トークンの候補は、全トークンの対数確率を合計する。